# 05 — Policy: autoria assistida e revisão

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 05](../docs/diagramas/svg/05_policy_autoria.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "05")
print({"lab": "05", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Ler a geração assistida criada pelo CloudFormation

A fase policy-draft solicita um rascunho ao serviço. Ela não publica uma Policy. Compare a linguagem natural com o Cedar gerado e procure permissões mais amplas que a intenção.

In [ ]:
environment = lab.outputs("policy-draft")
control = lab.aws.client("bedrock-agentcore-control")
generation = control.get_policy_generation(
    policyEngineId=lab.gateway["PolicyEngineId"],
    policyGenerationId=environment["PolicyGenerationId"],
)
assert generation["status"] == "GENERATED"
print(json.dumps(generation, indent=2, default=str))
assets, cursor = [], None
while True:
    page = control.list_policy_generation_assets(
        policyEngineId=lab.gateway["PolicyEngineId"],
        policyGenerationId=environment["PolicyGenerationId"],
        **({"nextToken": cursor} if cursor else {}),
    )
    assets.extend(page["policyGenerationAssets"])
    cursor = page.get("nextToken")
    if not cursor:
        break
assert assets, "O estado GENERATED precisa ter um rascunho para revisão"
for asset in assets:
    definition = asset["definition"]
    statement = definition.get("cedar", definition.get("policy", {})).get("statement")
    assert statement
    print({"fragment": asset.get("rawTextFragment"), "findings": asset.get("findings", [])})
    display(Code(statement, language="text"))

### Comparar políticas ativas com as declarações revisadas

A geração assistida não deve alterar permissões ativas. Consultamos cada Policy do stack bindings e comparamos o statement Cedar com o contrato gerado pelo IaC.

In [ ]:
from santander_demo.aws_workshop.contracts import policies
expected = policies(lab.gateway["GatewayArn"], lab.identity["Issuer"],
                    lab.identity["SupervisorClientId"], lab.identity["RiscosClientId"],
                    {x: lab.identity[f"User{x.upper()}Oid"] for x in "abc"})
base = dict(expected)
targets, cursor = [], None
while True:
    page = control.list_gateway_targets(
        gatewayIdentifier=lab.gateway["GatewayId"],
        **({"nextToken": cursor} if cursor else {}),
    )
    targets.extend(x["name"] for x in page["items"])
    cursor = page.get("nextToken")
    if not cursor:
        break
for domain in ["Retail", "Transaction", "Lending"]:
    if domain.lower() in targets:
        expected.update({domain + name: statement.replace(
            'AgentCore::Action::"', f'AgentCore::Action::"{domain.lower()}___')
            for name, statement in base.items()})
active, cursor = {}, None
while True:
    page = control.list_policies(
        policyEngineId=lab.gateway["PolicyEngineId"],
        **({"nextToken": cursor} if cursor else {}),
    )
    active.update({item["name"]: item for item in page["policies"]})
    cursor = page.get("nextToken")
    if not cursor:
        break
assert set(active) == set(expected), "Políticas fora do contrato IaC"
observed = {}
for name, statement in expected.items():
    policy = control.get_policy(
        policyEngineId=lab.gateway["PolicyEngineId"],
        policyId=active[name]["policyId"],
    )
    assert policy["status"] == "ACTIVE" and policy["enforcementMode"] == "ACTIVE"
    assert policy["definition"]["cedar"]["statement"].strip() == statement.strip()
    observed[name] = {"policy_id": policy["policyId"], "matches_iac": True}
print(json.dumps(observed, indent=2))

### Mostrar a operação que produz o rascunho

Este é o código executado pelo provider CloudFormation. A publicação é uma mudança separada no template bindings, com validação e revisão; o notebook não publica sugestões do modelo.

In [ ]:
print((ROOT / "infra/providers/policy_draft_handler.py").read_text())
print((ROOT / "infra/workshop/templates/bindings.json").read_text())

**Aceite:** geração em GENERATED e políticas ativas ainda iguais às declaradas no IaC. **Entra ID:** altere as condições de identidade para os claims verificados do token Entra antes de publicar.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.